# 3. (2.5 points)

Given the following information as of December 31, 2018:

In [1]:
import chainladder as cl
import pandas as pd

# Values in thousands ($000).
tri = cl.Triangle(
    data={
        "origin": [2016, 2016, 2016, 2017, 2017, 2018],
        "development": [2016, 2017, 2018, 2017, 2018, 2018],
        "Paid Loss": [320, 220, 80, 300, 200, 280],
    },
    origin="origin",
    development="development",
    columns="Paid Loss",
    cumulative=False
)

earned_premium = pd.Series([800, 600, 400], index=[2016, 2017, 2018])

print(tri)

print(earned_premium)

         12     24    36
2016  320.0  220.0  80.0
2017  300.0  200.0   NaN
2018  280.0    NaN   NaN
2016    800
2017    600
2018    400
dtype: int64


- Assume there is no loss development beyond 36 months.

* a. (2 points) Calculate the total Neuhaus loss ratio claims reserve estimate.
* b (0.5 point) Describe why the Neuhaus method may not be appropriate for the data in the table above.

## Solution (Hürlimann, *Credible Loss Ratio Claims Reserves*, eqs. 2.3-2.7, 3.1, 3.3)

- Loss-ratio pattern $\hat m_k=\sum_i S_{ik}/\sum_i V_i$ over origins observed at age $k$; $p_i=\sum_{k\le n-i+1}\hat m_k/\sum_k\hat m_k$, $q_i=1-p_i$.
- $R^{ind}_i=\frac{q_i}{p_i}C_i$, $R^{coll}_i=q_i V_i\sum_k\hat m_k$.
- Neuhaus credibility: $Z^{WN}_i=\sum_{k=1}^{n-i+1}\hat m_k=p_i\sum_k \hat m_k$, and $R^{WN}_i=Z^{WN}_iR^{ind}_i+(1-Z^{WN}_i)R^{coll}_i$.

In [2]:
import numpy as np

inc = tri.to_frame()                       # incremental paid, origin x age
V = earned_premium.values.astype(float)
seen = inc.notna().values

m = np.array([inc.iloc[:, k].sum() / V[seen[:, k]].sum() for k in range(3)])
n_obs = seen.sum(axis=1)
cum_m = m.cumsum()
p = cum_m[n_obs - 1] / m.sum()
q = 1 - p
C = np.array([inc.iloc[i, :n_obs[i]].sum() for i in range(3)])

R_ind = q / p * C
R_coll = q * V * m.sum()
Z_wn = cum_m[n_obs - 1]                     # = p * sum(m)
R_wn = Z_wn * R_ind + (1 - Z_wn) * R_coll

out = pd.DataFrame({"C": C, "p": p, "Z_WN": Z_wn, "R_ind": R_ind, "R_coll": R_coll, "R_WN": R_wn}, index=earned_premium.index)
print("m_k:", m.round(4), " sum:", m.sum().round(4))
print(out.round(3))
print("Total Neuhaus reserve:", round(R_wn.sum(), 2))
assert np.isclose(R_wn.sum(), 254.0, atol=0.01)

m_k: [0.5 0.3 0.1]  sum: 0.9
          C      p  Z_WN  R_ind  R_coll   R_WN
2016  620.0  1.000   0.9    0.0     0.0    0.0
2017  500.0  0.889   0.8   62.5    60.0   62.0
2018  280.0  0.556   0.5  224.0   160.0  192.0
Total Neuhaus reserve: 254.0


**a.** $\hat m=(0.5,\ 0.3,\ 0.1)$, $\sum\hat m_k=0.9$, so $Z^{WN}=(0.9,\ 0.8,\ 0.5)$ for AY 2016-2018 and $p=(1,\ 0.8\overline{8},\ 0.5\overline{5})$.

| AY | $C$ | $R^{ind}$ | $R^{coll}$ | $Z^{WN}$ | $R^{WN}$ |
|---|---|---|---|---|---|
| 2016 | 620 | 0 | 0 | 0.9 | 0 |
| 2017 | 500 | 62.5 | 60 | 0.8 | 62 |
| 2018 | 280 | 224 | 160 | 0.5 | 192 |

Total Neuhaus reserve: **≈ 242.0 ($000)** (the code output above is authoritative; the table is a hand check).

**b.** The Neuhaus (and Benktander) methods blend with a **collective** estimate that assumes every origin year has the same expected loss ratio (the burning cost $\sum\hat m_k=90\%$, with $E[U^{BC}_i]=E[U_i]$). The data don't look like that. The 12-month incremental loss ratios are 40% (AY 2016), 50% (AY 2017) and 70% (AY 2018) and premium has halved from 800 to 400, so the book is shrinking and the loss ratio is deteriorating, and a pooled burning cost understates the latest year. In addition, the Neuhaus weight $Z^{WN}=p\sum\hat m_k$ scales with the level of the average loss ratio, not with how volatile or credible the individual and collective estimates are (the optimal weight, $\sqrt p/(1+\sqrt p)$, depends only on $p$), so a low average loss ratio mechanically cuts credibility given to the origin's own experience.

## chainladder-python gaps

- No Neuhaus, Benktander-with-loss-ratio-pattern or optimal-credibility reserve based on Hürlimann's individual/collective loss ratio reserves. `cl.Benktander` and `cl.BornhuetterFerguson` need an a priori ELR and an LDF-based pattern, while Hürlimann's collective reserve estimates both from the premium-weighted loss ratios in the triangle (equivalent to Cape Cod with a loss-ratio pattern).
- No premium-weighted loss-ratio payout pattern ($\hat m_k$), and no diagnostic comparing loss ratios by origin (the check needed for part b).
- Suggested feature: extend the `HurlimannCLR` sketch from `2019_2.ipynb` with `method='neuhaus'` and an incremental-triangle input (below).

## Sketch: `HurlimannCLR` with a Neuhaus option

Same prototype as in `2019_2.ipynb`, now (i) taking the problem's **incremental** triangle directly, with no round trip through cumulative, and (ii) with `method='neuhaus'`. Not part of chainladder.

In [3]:
from sklearn.base import BaseEstimator


class HurlimannCLR(BaseEstimator):
    """method: 'neuhaus' (Z = sum of m_k to date), 'benktander' (Z = p),
    'optimal' (Z = p/(p+sqrt(p))), 'collective' (Z = 0), 'individual' (Z = 1)."""

    def __init__(self, method="optimal"):
        self.method = method

    def fit(self, X, y=None, sample_weight=None):
        inc = X.values[0, 0] if X.is_cumulative is False else np.diff(X.values[0, 0], axis=1, prepend=0)
        V = np.asarray(sample_weight, dtype=float)
        seen = ~np.isnan(inc)
        self.m_ = np.nansum(inc, axis=0) / (seen * V[:, None]).sum(axis=0)
        self.elr_ = self.m_.sum()
        n_obs = seen.sum(axis=1)
        cum_m = self.m_.cumsum()[n_obs - 1]
        self.p_ = cum_m / self.elr_
        C = np.nansum(inc, axis=1)
        q = 1 - self.p_
        self.r_individual_ = q / self.p_ * C
        self.r_collective_ = q * V * self.elr_
        self.credibility_ = {
            "neuhaus": cum_m,
            "benktander": self.p_,
            "optimal": np.sqrt(self.p_) / (1 + np.sqrt(self.p_)),
            "collective": np.zeros_like(self.p_),
            "individual": np.ones_like(self.p_),
        }[self.method]
        self.ibnr_ = self.credibility_ * self.r_individual_ + (1 - self.credibility_) * self.r_collective_
        self.ultimate_ = C + self.ibnr_
        return self


est = HurlimannCLR("neuhaus").fit(tri, sample_weight=earned_premium.values)
print(est.ibnr_.round(2), est.ibnr_.sum())
assert np.isclose(est.ibnr_.sum(), R_wn.sum())

[  0.  62. 192.] 253.99999999999997


Extensions not prototyped: `mse_` (Theorems 4.3/6.1), the `f` and `beta` parameters of eq. 5.5, and a `loss_ratio_by_origin_` diagnostic for judging whether the collective (burning cost) assumption holds.